# LightFM Baseline dưới thiết lập Leave-One-Out

Notebook này triển khai **LightFM baseline** làm mốc so sánh cho các thử nghiệm recommendation trong bài báo. Mô hình sử dụng **LightFM** với loss = **WARP** trong thiết lập **implicit feedback**, trong đó rating được nhị phân hóa trước khi huấn luyện.

## Thiết kế thực nghiệm
- **Leave-One-Out (LOO) theo người dùng:** với mỗi user, tương tác mới nhất được dùng cho test, tương tác kế tiếp dùng cho validation, phần còn lại dùng cho train.
- **Biểu diễn tương tác nhị phân:** `rate >= POSITIVE_THRESHOLD` được xem là positive; các tương tác còn lại không được đưa vào ma trận huấn luyện.
- **Loại trừ lịch sử train khi xếp hạng:** trong đánh giá Top-$K$, các item đã xuất hiện trong train của user sẽ được mask.
- **Đánh giá warm/cold trên tập test:** kết quả được tách theo item đã xuất hiện trong train và item chưa xuất hiện trong train.

## Nhóm đặc trưng sử dụng
| Nhóm | Thành phần | Vai trò |
|---|---|---|
| **Item features** | `genre`, `topic`, `country` (categorical) + `year`, `duration` (continuous) | Cung cấp tín hiệu nội dung có cấu trúc với độ phủ tốt và độ thưa thấp |
| **User features** | Metadata hồ sơ tĩnh từ `users_df` | Cung cấp mức cá nhân hóa cơ bản mà không sử dụng lịch sử tương tác |

## Mục 0 - Thiết lập môi trường và cấu hình

Khai báo nguồn dữ liệu, thư mục checkpoint và toàn bộ hyperparameter chính, bao gồm ngưỡng implicit feedback, số latent factors, learning rate, regularization và danh sách giá trị $K$ dùng để đánh giá.

In [ ]:
import os, pickle, ast, warnings
from collections import Counter

import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Dataset URLs ───────────────────────────────────────────────────────────────
MOVIES_URL  = "https://raw.githubusercontent.com/lynchblue/movie-rating-dataset/main/data/movies.csv"
RATINGS_URL = "https://raw.githubusercontent.com/lynchblue/movie-rating-dataset/main/data/ratings.csv"
USERS_URL   = "https://raw.githubusercontent.com/lynchblue/movie-rating-dataset/main/data/user_profiles.csv"

CHECKPOINT_DIR = "checkpoints_baseline"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── Configuration ──────────────────────────────────────────────────────────────
POSITIVE_THRESHOLD = 7       # Binary implicit: rate >= này → positive
MIN_RATINGS        = 10      # Lọc user có ít hơn N ratings (cần >= 3 để LOO hợp lệ)

# LightFM
NO_COMPONENTS  = 64
LEARNING_RATE  = 0.05
ITEM_ALPHA     = 1e-6
USER_ALPHA     = 1e-6
MAX_EPOCHS     = 30
PATIENCE       = 5
NUM_THREADS    = 4
K_VALUES       = (5, 10, 20, 50)

print(f"[BASELINE] Config: threshold={POSITIVE_THRESHOLD}, components={NO_COMPONENTS}, "
      f"lr={LEARNING_RATE}, patience={PATIENCE}")
print(f"[BASELINE] Split : Leave-One-Out (LOO) per user")
print(f"[BASELINE] Features: Tier 1 (genre/topic/country/year/duration) + Tier 3 (user metadata)")
print(f"[BASELINE] NO Tier 2 (TinyBERT / KGE / PCA)")

[BASELINE] Config: threshold=7, components=64, lr=0.05, patience=5
[BASELINE] Split : Leave-One-Out (LOO) per user
[BASELINE] Features: Tier 1 (genre/topic/country/year/duration) + Tier 3 (user metadata)
[BASELINE] NO Tier 2 (TinyBERT / KGE / PCA)


## Mục 0b - Các hàm hỗ trợ

Nhóm hàm này thực hiện các thao tác tiền xử lý dùng chung trong toàn bộ pipeline, chẳng hạn chuẩn hóa văn bản, tách token theo dấu `|`, lọc token hiếm và làm sạch trường `plot`.

In [ ]:
def cleanup_column(series):
    """Fix encoding artefacts in pipe-separated string Series."""
    replacements = {
        '| ': '|', '\xc3\xa9': 'é', '\xc3\x81': 'Á', '\xc3\x93': 'Ó',
        '\xc3\xa1': 'á', '\xc3\xb3': 'ó', '\xc3\xb1': 'ñ', '\xc3\xad': 'í',
        '\xc3\xba': 'ú', 'Ã©': 'é', 'Ã¡': 'á', 'Ã³': 'ó', 'Ã±': 'ñ',
        'Ãº': 'ú',
    }
    col = series.copy()
    for bad, good in replacements.items():
        col = col.str.replace(bad, good, regex=False)
    return col


def tokenize(series, sep='|'):
    """Split pipe-separated Series → list-of-lists. NaN → []."""
    return [
        [t.strip() for t in s.split(sep) if t.strip()]
        if isinstance(s, str) else []
        for s in series
    ]


def filter_rare(token_lists, min_count):
    """Loại token xuất hiện < min_count lần."""
    if min_count <= 1:
        return token_lists
    counter = Counter()
    for toks in token_lists:
        counter.update(set(toks))
    keep = {t for t, c in counter.items() if c >= min_count}
    return [[t for t in toks if t in keep] for toks in token_lists]


def fix_plot(p):
    """Fix byte-string encoded plots và encoding artifacts."""
    if not isinstance(p, str) or p.strip() in ('', 'nan'):
        return ""
    if p.startswith("b'") or p.startswith('b"'):
        try:
            decoded = ast.literal_eval(p)
            if isinstance(decoded, bytes):
                return decoded.decode('utf-8', errors='replace')
        except Exception:
            try:
                decoded = ast.literal_eval(p)
                if isinstance(decoded, bytes):
                    return decoded.decode('latin-1', errors='replace')
            except Exception:
                return p[2:-1]
    return p


print("Helpers defined: cleanup_column, tokenize, filter_rare, fix_plot")

Helpers defined: cleanup_column, tokenize, filter_rare, fix_plot


## Mục 1 - Nạp dữ liệu

Tải `movies.csv`, `ratings.csv`, `user_profiles.csv` và chuẩn hóa cột thời gian để phục vụ chia dữ liệu theo Leave-One-Out.

In [ ]:
ratings_df = pd.read_csv(RATINGS_URL)
users_df   = pd.read_csv(USERS_URL)
movies_df  = pd.read_csv(MOVIES_URL)

ratings_df["date"] = pd.to_datetime(ratings_df["date"])

print(f"Users   : {users_df['id'].nunique()}")
print(f"Movies  : {movies_df['id'].nunique()}")
print(f"Ratings : {len(ratings_df):,}")
print(f"Rating range: {ratings_df['rate'].min()} – {ratings_df['rate'].max()}")
print(f"Date range  : {ratings_df['date'].min().date()} – {ratings_df['date'].max().date()}")

Users   : 482
Movies  : 78628
Ratings : 1,172,038
Rating range: 1 – 10
Date range  : 2002-08-14 – 2021-04-01


## Mục 2 - Lọc dữ liệu hợp lệ và kiểm tra toàn vẹn

Loại bỏ các bản ghi không hợp lệ để bảo đảm LOO split khả thi và tránh lỗi do user hoặc movie không khớp giữa các bảng dữ liệu.

In [ ]:
# 2.1 Lọc orphan users (có rating nhưng KHÔNG có profile)
users_in_profile = set(users_df["id"].unique())
ratings_df = ratings_df[ratings_df["id"].isin(users_in_profile)].copy()
print(f"Sau lọc orphan users    : {len(ratings_df):,} ratings")

# 2.2 Lọc user có ít hơn MIN_RATINGS ratings
rating_counts = ratings_df.groupby("id")["movie_id"].count()
valid_users   = rating_counts[rating_counts >= MIN_RATINGS].index
ratings_df    = ratings_df[ratings_df["id"].isin(valid_users)].copy()
users_df      = users_df[users_df["id"].isin(valid_users)].copy()
print(f"Sau lọc min {MIN_RATINGS} ratings : {len(ratings_df):,} ratings, {ratings_df['id'].nunique()} users")

# 2.3 Lọc movie không tồn tại trong movies.csv
movies_in_table = set(movies_df["id"].unique())
ratings_df = ratings_df[ratings_df["movie_id"].isin(movies_in_table)].copy()
print(f"Sau lọc missing movies  : {len(ratings_df):,} ratings")

Sau lọc orphan users    : 963,266 ratings
Sau lọc min 10 ratings : 963,231 ratings, 474 users
Sau lọc missing movies  : 963,228 ratings


## Mục 3 - Chuẩn hóa các cột văn bản

Làm sạch lại encoding trước khi token hóa. Do metadata là thuộc tính tĩnh của item, bước này không đưa thông tin từ validation/test vào quá trình huấn luyện theo cách gây data leakage.

In [ ]:
# 3.1 Fix encoding artifacts cho pipe-separated columns
for col in ["country_name", "genres", "topics", "directors", "actors",
            "script", "producer", "music", "photography"]:
    movies_df[col] = cleanup_column(movies_df[col])

# 3.2 Fix byte-string plots (không dùng cho embedding, chỉ cleanup)
movies_df["plot_clean"] = movies_df["plot"].apply(fix_plot)

print("Text cleanup hoàn tất.")
print(f"Plot NaN/empty: {(movies_df['plot_clean'] == '').sum():,} / {len(movies_df):,}")

Text cleanup hoàn tất.
Plot NaN/empty: 2,442 / 78,628


## Mục 4 - Chia dữ liệu theo Leave-One-Out (LOO)

Với mỗi user, các rating được sắp xếp theo thời gian và chia như sau:
- **Test:** tương tác mới nhất
- **Validation:** tương tác ngay trước test
- **Train:** toàn bộ phần còn lại

Điều kiện `MIN_RATINGS >= 3` bảo đảm mỗi user đều có đủ dữ liệu cho cả train, validation và test.

In [ ]:
ratings_df = ratings_df.sort_values(["id", "date"]).reset_index(drop=True)

train_idx = []
valid_idx = []
test_idx  = []

for uid, group in ratings_df.groupby("id", sort=False):
    idx = group.index.tolist()
    if len(idx) < 3:
        train_idx.extend(idx)
        continue
    test_idx.append(idx[-1])
    valid_idx.append(idx[-2])
    train_idx.extend(idx[:-2])

train_df = ratings_df.loc[train_idx].copy().reset_index(drop=True)
valid_df = ratings_df.loc[valid_idx].copy().reset_index(drop=True)
test_df  = ratings_df.loc[test_idx].copy().reset_index(drop=True)

print(f"Train : {len(train_df):>7,} ratings  |  {train_df['id'].nunique()} users")
print(f"Valid : {len(valid_df):>7,} ratings  |  {valid_df['id'].nunique()} users  (1 rating/user)")
print(f"Test  : {len(test_df):>7,} ratings  |  {test_df['id'].nunique()} users  (1 rating/user)")
print(f"\nDate range train: {train_df['date'].min().date()} – {train_df['date'].max().date()}")
print(f"Date range valid: {valid_df['date'].min().date()} – {valid_df['date'].max().date()}")
print(f"Date range test : {test_df['date'].min().date()} – {test_df['date'].max().date()}")

Train : 962,280 ratings  |  474 users
Valid :     474 ratings  |  474 users  (1 rating/user)
Test  :     474 ratings  |  474 users  (1 rating/user)

Date range train: 2002-08-14 – 2021-03-31
Date range valid: 2011-12-06 – 2021-04-01
Date range test : 2011-12-08 – 2021-04-01


## Mục 5 - Phân loại item warm và cold trên tập kiểm thử

Phân loại dựa trên việc item có xuất hiện trong **train** hay không:
- **Warm item:** đã xuất hiện trong train.
- **Cold item:** chưa xuất hiện trong train.

Thiết lập này giúp đánh giá riêng khả năng mô hình xử lý kịch bản item mới, nơi vai trò của item features trở nên quan trọng hơn.

In [ ]:
movies_in_train = set(train_df["movie_id"].unique())

test_warm_mask = test_df["movie_id"].isin(movies_in_train)
test_cold_mask = ~test_warm_mask

test_warm_df = test_df[test_warm_mask].copy().reset_index(drop=True)
test_cold_df = test_df[test_cold_mask].copy().reset_index(drop=True)

print(f"Unique items in train : {len(movies_in_train):,}")
print(f"Unique items in valid : {valid_df['movie_id'].nunique():,}")
print(f"Unique items in test  : {test_df['movie_id'].nunique():,}")
print()
print(f"TEST total : {len(test_df):>6,} ratings")
print(f"TEST warm  : {len(test_warm_df):>6,} ratings  "
      f"({len(test_warm_df)/len(test_df)*100:.1f}%)  "
      f"— {test_warm_df['movie_id'].nunique():,} unique movies")
print(f"TEST cold  : {len(test_cold_df):>6,} ratings  "
      f"({len(test_cold_df)/len(test_df)*100:.1f}%)  "
      f"— {test_cold_df['movie_id'].nunique():,} unique movies")

Unique items in train : 75,115
Unique items in valid : 398
Unique items in test  : 407

TEST total :    474 ratings
TEST warm  :    446 ratings  (94.1%)  — 379 unique movies
TEST cold  :     28 ratings  (5.9%)  — 28 unique movies


## Mục 6 - Xây dựng item features cơ bản

Sử dụng các đặc trưng nội dung có độ phủ tốt và ít nhiễu:
- **Categorical:** `genre`, `topic`, `country`
- **Continuous:** `year`, `duration`

Baseline này không sử dụng dense embeddings và không đưa trực tiếp `actor` hoặc `director` vào dạng one-hot để tránh không gian đặc trưng quá thưa.

In [ ]:
# Tokenize categorical columns
genre_toks   = tokenize(movies_df["genres"])
topic_toks   = tokenize(movies_df["topics"])
country_toks = filter_rare(tokenize(movies_df["country_name"]), min_count=2)

# Continuous normalization params
year_min   = movies_df["year_published"].min()
year_range = movies_df["year_published"].max() - year_min
dur_cap    = movies_df["duration"].quantile(0.99)

# Build item feature dicts: {movie_id: {feature_name: weight}}
item_feat_dicts = {}
for i in range(len(movies_df)):
    row = movies_df.iloc[i]
    mid = int(row["id"])
    feats = {}

    # ── Continuous features (normalized weight) ──
    if pd.notna(row["year_published"]) and year_range > 0:
        feats["year"] = (row["year_published"] - year_min) / year_range
    if pd.notna(row["duration"]) and dur_cap > 0:
        feats["duration"] = min(row["duration"] / dur_cap, 1.0)

    # ── Categorical features (weight = 1.0) ──
    for g in genre_toks[i]:
        feats[f"genre:{g}"] = 1.0
    for t in topic_toks[i]:
        feats[f"topic:{t}"] = 1.0
    for c in country_toks[i]:
        feats[f"country:{c}"] = 1.0

    item_feat_dicts[mid] = feats

# Summary
all_item_tags = set()
for feats in item_feat_dicts.values():
    all_item_tags.update(feats.keys())

n_genre   = len({t for t in all_item_tags if t.startswith("genre:")})
n_topic   = len({t for t in all_item_tags if t.startswith("topic:")})
n_country = len({t for t in all_item_tags if t.startswith("country:")})
n_cont    = len({t for t in all_item_tags if t in ("year", "duration")})

print(f"[BASELINE] Item features (Tier 1 only): {len(all_item_tags)} unique tags")
print(f"  genre     : {n_genre}")
print(f"  topic     : {n_topic}")
print(f"  country   : {n_country}")
print(f"  continuous: {n_cont} (year, duration)")
print(f"\n[BASELINE] Không có Tier 2 (TinyBERT / KGE / PCA) — tổng item features = {len(all_item_tags)}")

[BASELINE] Item features (Tier 1 only): 590 unique tags
  genre     : 19
  topic     : 395
  country   : 174
  continuous: 2 (year, duration)

[BASELINE] Không có Tier 2 (TinyBERT / KGE / PCA) — tổng item features = 590


## Mục 7 - Xây dựng user features từ hồ sơ tĩnh

User features được tạo từ metadata hồ sơ tĩnh trong `users_df`, không phụ thuộc vào lịch sử tương tác:
- Binary flags như `night_owl`, `early_bird`, ...
- Các biến đã binned như preferred hour hoặc account creation year
- Các tag theo preferred weekday

Thiết kế này giữ cho baseline đủ đơn giản nhưng vẫn phản ánh mức cá nhân hóa tối thiểu.

In [ ]:
def bin_preferred_hour(h):
    if pd.isna(h):       return "hour:unknown"
    if 5 <= h <= 11:     return "hour:morning"
    elif 12 <= h <= 17:  return "hour:afternoon"
    elif 18 <= h <= 22:  return "hour:evening"
    else:                return "hour:night"

def bin_account_year(y):
    if pd.isna(y):     return "acc:unknown"
    if y < 2010:       return "acc:pre2010"
    elif y <= 2013:    return "acc:2010_2013"
    else:              return "acc:post2013"

users_df["hour_bin"]     = users_df["preferred_hour"].apply(bin_preferred_hour)
users_df["acc_year_bin"] = users_df["account_creation_year"].apply(bin_account_year)
users_df["weekday_bin"]  = "weekday:" + users_df["preferred_weekday"].astype(str)

print("Hour bins    :", users_df["hour_bin"].value_counts().to_dict())
print("Acc year bins:", users_df["acc_year_bin"].value_counts().to_dict())
print("Weekday bins :", users_df["weekday_bin"].value_counts().to_dict())

BINARY_FLAGS = ["night_owl", "early_bird", "weekend_tweeter", "week_tweeter", "geo_enabled"]

user_feat_dicts = {}
for _, row in users_df.iterrows():
    uid   = int(row["id"])
    feats = {}

    for col in BINARY_FLAGS:
        if row.get(col, 0) == 1:
            feats[f"user:{col}"] = 1.0

    feats[row["hour_bin"]]     = 1.0
    feats[row["acc_year_bin"]] = 1.0
    feats[row["weekday_bin"]]  = 1.0

    user_feat_dicts[uid] = feats

all_user_tags = set()
for feats in user_feat_dicts.values():
    all_user_tags.update(feats.keys())

n_binary  = len({t for t in all_user_tags if t.startswith("user:")})
n_hour    = len({t for t in all_user_tags if t.startswith("hour:")})
n_acc     = len({t for t in all_user_tags if t.startswith("acc:")})
n_weekday = len({t for t in all_user_tags if t.startswith("weekday:")})

print(f"\n[BASELINE] Tier 3 user features: {len(all_user_tags)} unique tags")
print(f"  binary flags : {n_binary}")
print(f"  hour bins    : {n_hour}")
print(f"  acc year bins: {n_acc}")
print(f"  weekday bins : {n_weekday}")
print(f"  Users với features: {len(user_feat_dicts)} / {users_df['id'].nunique()}")
print(f"\nSample user (id={list(user_feat_dicts.keys())[0]}): {list(user_feat_dicts.values())[0]}")

Hour bins    : {'hour:evening': 179, 'hour:afternoon': 127, 'hour:night': 101, 'hour:morning': 67}
Acc year bins: {'acc:2010_2013': 349, 'acc:pre2010': 73, 'acc:post2013': 52}
Weekday bins : {'weekday:0': 119, 'weekday:6': 90, 'weekday:2': 80, 'weekday:3': 64, 'weekday:1': 57, 'weekday:4': 40, 'weekday:5': 24}

[BASELINE] Tier 3 user features: 19 unique tags
  binary flags : 5
  hour bins    : 4
  acc year bins: 3
  weekday bins : 7
  Users với features: 474 / 474

Sample user (id=103007): {'user:night_owl': 1.0, 'user:geo_enabled': 1.0, 'hour:evening': 1.0, 'acc:pre2010': 1.0, 'weekday:1': 1.0}


## Mục 8 - Lưu checkpoint của baseline

Lưu các artifacts sau preprocessing, bao gồm dataframes, feature dictionaries và cấu hình, để tái lập nhanh mà không cần chạy lại toàn bộ phần chuẩn bị dữ liệu.

In [ ]:
BASELINE_CHECKPOINT = os.path.join(CHECKPOINT_DIR, "baseline_preprocessed.pkl")

baseline_artifacts = {
    # DataFrames
    "ratings_df":    ratings_df,
    "users_df":      users_df,
    "movies_df":     movies_df,
    "train_df":      train_df,
    "valid_df":      valid_df,
    "test_df":       test_df,
    "test_warm_df":  test_warm_df,
    "test_cold_df":  test_cold_df,
    # Feature dicts
    "item_feat_dicts": item_feat_dicts,
    "user_feat_dicts": user_feat_dicts,
    "all_item_tags":   all_item_tags,
    "all_user_tags":   all_user_tags,
    # Config
    "config": {
        "POSITIVE_THRESHOLD": POSITIVE_THRESHOLD,
        "MIN_RATINGS":        MIN_RATINGS,
        "NO_COMPONENTS":      NO_COMPONENTS,
        "LEARNING_RATE":      LEARNING_RATE,
        "ITEM_ALPHA":         ITEM_ALPHA,
        "USER_ALPHA":         USER_ALPHA,
        "MAX_EPOCHS":         MAX_EPOCHS,
        "PATIENCE":           PATIENCE,
        "NUM_THREADS":        NUM_THREADS,
        "K_VALUES":           K_VALUES,
    },
}

with open(BASELINE_CHECKPOINT, "wb") as f:
    pickle.dump(baseline_artifacts, f, protocol=pickle.HIGHEST_PROTOCOL)

size_mb = os.path.getsize(BASELINE_CHECKPOINT) / (1024 * 1024)
print(f"Saved baseline checkpoint: {BASELINE_CHECKPOINT}  ({size_mb:.1f} MB)")
print(f"Keys: {list(baseline_artifacts.keys())}")

Saved baseline checkpoint: checkpoints_baseline/baseline_preprocessed.pkl  (146.9 MB)
Keys: ['ratings_df', 'users_df', 'movies_df', 'train_df', 'valid_df', 'test_df', 'test_warm_df', 'test_cold_df', 'item_feat_dicts', 'user_feat_dicts', 'all_item_tags', 'all_user_tags', 'config']


## Mục 8b - Nạp checkpoint của baseline

Đặt `LOAD_BASELINE = True` để bỏ qua các bước preprocessing trước đó và tiếp tục trực tiếp từ bước xây dựng `Dataset` của LightFM.

In [ ]:
import os, pickle, warnings
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore", category=FutureWarning)

LOAD_BASELINE      = False   # Đặt True nếu đã có checkpoint
CHECKPOINT_DIR     = "checkpoints_baseline"
BASELINE_CHECKPOINT = os.path.join(CHECKPOINT_DIR, "baseline_preprocessed.pkl")

if LOAD_BASELINE and os.path.exists(BASELINE_CHECKPOINT):
    with open(BASELINE_CHECKPOINT, "rb") as f:
        art = pickle.load(f)

    ratings_df      = art["ratings_df"]
    users_df        = art["users_df"]
    movies_df       = art["movies_df"]
    train_df        = art["train_df"]
    valid_df        = art["valid_df"]
    test_df         = art["test_df"]
    test_warm_df    = art["test_warm_df"]
    test_cold_df    = art["test_cold_df"]
    item_feat_dicts = art["item_feat_dicts"]
    user_feat_dicts = art["user_feat_dicts"]
    all_item_tags   = art["all_item_tags"]
    all_user_tags   = art["all_user_tags"]

    cfg = art["config"]
    POSITIVE_THRESHOLD = cfg["POSITIVE_THRESHOLD"]
    MIN_RATINGS        = cfg["MIN_RATINGS"]
    NO_COMPONENTS      = cfg["NO_COMPONENTS"]
    LEARNING_RATE      = cfg["LEARNING_RATE"]
    ITEM_ALPHA         = cfg["ITEM_ALPHA"]
    USER_ALPHA         = cfg["USER_ALPHA"]
    MAX_EPOCHS         = cfg["MAX_EPOCHS"]
    PATIENCE           = cfg["PATIENCE"]
    NUM_THREADS        = cfg["NUM_THREADS"]
    K_VALUES           = cfg["K_VALUES"]

    print(f"Loaded baseline checkpoint from: {BASELINE_CHECKPOINT}")
    print(f"  Train     : {len(train_df):>7,} | {train_df['id'].nunique()} users")
    print(f"  Valid     : {len(valid_df):>7,} | {valid_df['id'].nunique()} users")
    print(f"  Test      : {len(test_df):>7,} | {test_df['id'].nunique()} users")
    print(f"  Test warm : {len(test_warm_df):>7,} | {test_warm_df['id'].nunique()} users")
    print(f"  Test cold : {len(test_cold_df):>7,} | {test_cold_df['id'].nunique()} users")
    print(f"  Item features : {len(all_item_tags)} (Tier 1 only)")
    print(f"  User features : {len(all_user_tags)} (Tier 3)")
else:
    if LOAD_BASELINE:
        print(f"⚠ Checkpoint không tồn tại: {BASELINE_CHECKPOINT}")
        print("  Hãy chạy Bước 0→8 trước.")
    else:
        print("LOAD_BASELINE=False — dùng kết quả từ preprocessing ở trên.")

LOAD_BASELINE=False — dùng kết quả từ preprocessing ở trên.


## Mục 9 - Khởi tạo `Dataset` của LightFM

Gom toàn bộ tên feature của user và item, sau đó xây dựng các sparse feature matrices theo đúng API của LightFM.

In [ ]:
pip install lightfm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.4/316.4 kB 7.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lightfm: filename=lightfm-1.17-cp311-cp311-linux_x86_64.whl size=831125 sha256=2156d1d3f68d29eeb392557ccd3329379482b9e7da303999d286f0a7ee694b07
  Stored in directory: /root/.cache/pip/wheels/b9/0d/8a/0729d2e6e3ca2a898ba55201f905da7db3f838a33df5b3fcdd
Successfully built lightfm


In [ ]:
from lightfm import LightFM
from lightfm.data import Dataset

# ── Collect ALL feature names ──
all_item_tags = set()
for feats in item_feat_dicts.values():
    all_item_tags.update(feats.keys())

all_user_tags = set()
for feats in user_feat_dicts.values():
    all_user_tags.update(feats.keys())

# ── Fit Dataset ──
dataset = Dataset()
dataset.fit(
    users=users_df["id"].tolist(),
    items=movies_df["id"].tolist(),
    user_features=list(all_user_tags),
    item_features=list(all_item_tags),
)

# ── Build Feature Matrices ──
item_features_raw = [(mid, feats) for mid, feats in item_feat_dicts.items()]
user_features_raw = [(uid, feats) for uid, feats in user_feat_dicts.items()]

item_feature_matrix = dataset.build_item_features(item_features_raw, normalize=False)
user_feature_matrix = dataset.build_user_features(user_features_raw, normalize=False)

print(f"User feature matrix : {user_feature_matrix.shape}")
print(f"Item feature matrix : {item_feature_matrix.shape}")
print(f"Item feature density: {item_feature_matrix.nnz / np.prod(item_feature_matrix.shape) * 100:.4f}%")
print(f"\n[BASELINE] Total user features: {len(all_user_tags)} (Tier 3 only)")
print(f"[BASELINE] Total item features: {len(all_item_tags)} (Tier 1 only — no Tier 2)")

User feature matrix : (474, 493)
Item feature matrix : (78628, 79218)
Item feature density: 0.0096%

[BASELINE] Total user features: 19 (Tier 3 only)
[BASELINE] Total item features: 590 (Tier 1 only — no Tier 2)


## Mục 10 - Xây dựng interactions nhị phân

Quy ước implicit feedback:
- Nếu `rate >= POSITIVE_THRESHOLD` thì xem là positive với weight bằng 1.
- Nếu nhỏ hơn ngưỡng thì bỏ qua.

Notebook không sử dụng `sample_weight` để giữ đúng giả định của thiết lập WARP trong baseline này.

In [ ]:
def build_interactions_binary(df, dataset, threshold=POSITIVE_THRESHOLD):
    """Binary implicit: rate >= threshold → positive. Không trả sample_weight."""
    positives = df[df["rate"] >= threshold]
    if len(positives) == 0:
        return None
    pairs = [(int(r["id"]), int(r["movie_id"])) for _, r in positives.iterrows()]
    interactions, _ = dataset.build_interactions(pairs)
    return interactions

train_interactions = build_interactions_binary(train_df, dataset)
valid_inter        = build_interactions_binary(valid_df, dataset)
test_inter         = build_interactions_binary(test_df, dataset)
test_warm_inter    = build_interactions_binary(test_warm_df, dataset)
test_cold_inter    = build_interactions_binary(test_cold_df, dataset)

def nnz(m): return m.nnz if m is not None else 0

print(f"Threshold = {POSITIVE_THRESHOLD} (binary implicit, no sample_weight)")
print(f"Train interactions : {nnz(train_interactions):,}")
print(f"Valid interactions : {nnz(valid_inter):,}")
print(f"Test  interactions : {nnz(test_inter):,}")
print(f"  ├─ Warm items    : {nnz(test_warm_inter):,}")
print(f"  └─ Cold items    : {nnz(test_cold_inter):,}")

Threshold = 7 (binary implicit, no sample_weight)
Train interactions : 412,438
Valid interactions : 240
Test  interactions : 262
  ├─ Warm items    : 253
  └─ Cold items    : 9


## Mục 11 - Các chỉ số đánh giá Top-$K$

Định nghĩa Precision@K, Recall@K, NDCG@K, HR@K và MRR@K. Trong quá trình xếp hạng, các item đã xuất hiện trong train của user được mask để tránh gợi ý lại item đã tương tác.

In [ ]:
def evaluate_metrics(model, test_interactions, train_interactions,
                     user_features, item_features,
                     k_values=(5, 10, 20), num_threads=1):
    """
    Tính Precision@K, Recall@K, NDCG@K, HR@K, MRR@K cho nhiều K.
    Returns: dict {K: {metric: value}}
    """
    if test_interactions is None or test_interactions.nnz == 0:
        return {k: {"precision": 0., "recall": 0., "ndcg": 0.,
                     "hr": 0., "mrr": 0.} for k in k_values}

    test_csr  = test_interactions.tocsr()
    train_csr = train_interactions.tocsr()
    n_users, n_items = test_csr.shape
    max_k = max(k_values)

    accum = {k: {"precision": [], "recall": [], "ndcg": [],
                  "hr": [], "mrr": []} for k in k_values}

    for u in range(n_users):
        true_items = set(test_csr[u].indices.tolist())
        if not true_items:
            continue

        scores = model.predict(
            user_ids=u,
            item_ids=np.arange(n_items),
            user_features=user_features,
            item_features=item_features,
            num_threads=num_threads,
        )

        # Mask items đã có trong train
        train_items = train_csr[u].indices
        scores[train_items] = -np.inf

        top_idx = np.argpartition(-scores, max_k)[:max_k]
        top_idx = top_idx[np.argsort(-scores[top_idx])]

        relevance  = np.array([1.0 if i in true_items else 0.0 for i in top_idx])
        n_relevant = len(true_items)

        for k in k_values:
            rel_k  = relevance[:k]
            n_hits = float(rel_k.sum())

            precision = n_hits / k
            recall    = n_hits / n_relevant if n_relevant > 0 else 0.0
            hr        = 1.0 if n_hits > 0 else 0.0

            discounts = 1.0 / np.log2(np.arange(2, k + 2))
            dcg  = float((rel_k * discounts).sum())
            idcg = float(discounts[:min(n_relevant, k)].sum())
            ndcg = dcg / idcg if idcg > 0 else 0.0

            mrr = 0.0
            for j in range(k):
                if rel_k[j] > 0:
                    mrr = 1.0 / (j + 1)
                    break

            accum[k]["precision"].append(precision)
            accum[k]["recall"].append(recall)
            accum[k]["ndcg"].append(ndcg)
            accum[k]["hr"].append(hr)
            accum[k]["mrr"].append(mrr)

    summary = {}
    for k in k_values:
        summary[k] = {m: float(np.mean(v)) if v else 0.0
                      for m, v in accum[k].items()}
    return summary


def print_metrics(metrics, label):
    print(f"\n=== {label} ===")
    print(f"{'K':>4} | {'Prec':>8} | {'Recall':>8} | {'NDCG':>8} | {'HR':>8} | {'MRR':>8}")
    print("-" * 58)
    for k, m in sorted(metrics.items()):
        print(f"{k:>4} | {m['precision']:>8.4f} | {m['recall']:>8.4f} | "
              f"{m['ndcg']:>8.4f} | {m['hr']:>8.4f} | {m['mrr']:>8.4f}")


print("Metrics defined: evaluate_metrics, print_metrics")

Metrics defined: evaluate_metrics, print_metrics


## Mục 12 - Huấn luyện LightFM với WARP

Huấn luyện mô hình theo epoch, đánh giá trên `valid_inter` và áp dụng early stopping theo NDCG@10 để chọn checkpoint tốt nhất.

In [ ]:
model = LightFM(
    loss="warp",
    no_components=NO_COMPONENTS,
    learning_rate=LEARNING_RATE,
    item_alpha=ITEM_ALPHA,
    user_alpha=USER_ALPHA,
    random_state=42,
)

best_ndcg10      = -1.0
patience_counter = 0
history          = []

for epoch in range(1, MAX_EPOCHS + 1):
    model.fit_partial(
        interactions=train_interactions,
        user_features=user_feature_matrix,
        item_features=item_feature_matrix,
        epochs=1,
        num_threads=NUM_THREADS,
        verbose=False,
    )

    valid_metrics = evaluate_metrics(
        model, valid_inter, train_interactions,
        user_feature_matrix, item_feature_matrix,
        k_values=K_VALUES, num_threads=NUM_THREADS,
    )

    ndcg10 = valid_metrics[10]["ndcg"]
    history.append({
        "epoch": epoch,
        **{f"ndcg@{k}": valid_metrics[k]["ndcg"] for k in K_VALUES}
    })
    print(f"Epoch {epoch:>2} | "
          + " | ".join(f"NDCG@{k}: {valid_metrics[k]['ndcg']:.4f}" for k in K_VALUES))

    if ndcg10 > best_ndcg10:
        best_ndcg10      = ndcg10
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}")
            break

print(f"\n[BASELINE] Best Valid NDCG@10: {best_ndcg10:.4f}")

Epoch  1 | NDCG@5: 0.0068 | NDCG@10: 0.0096 | NDCG@20: 0.0117 | NDCG@50: 0.0166
Epoch  2 | NDCG@5: 0.0079 | NDCG@10: 0.0118 | NDCG@20: 0.0149 | NDCG@50: 0.0189
Epoch  3 | NDCG@5: 0.0089 | NDCG@10: 0.0103 | NDCG@20: 0.0144 | NDCG@50: 0.0195
Epoch  4 | NDCG@5: 0.0122 | NDCG@10: 0.0149 | NDCG@20: 0.0159 | NDCG@50: 0.0223
Epoch  5 | NDCG@5: 0.0116 | NDCG@10: 0.0130 | NDCG@20: 0.0150 | NDCG@50: 0.0241
Epoch  6 | NDCG@5: 0.0138 | NDCG@10: 0.0166 | NDCG@20: 0.0196 | NDCG@50: 0.0245
Epoch  7 | NDCG@5: 0.0133 | NDCG@10: 0.0171 | NDCG@20: 0.0171 | NDCG@50: 0.0260
Epoch  8 | NDCG@5: 0.0139 | NDCG@10: 0.0151 | NDCG@20: 0.0183 | NDCG@50: 0.0232
Epoch  9 | NDCG@5: 0.0146 | NDCG@10: 0.0186 | NDCG@20: 0.0207 | NDCG@50: 0.0290
Epoch 10 | NDCG@5: 0.0126 | NDCG@10: 0.0153 | NDCG@20: 0.0194 | NDCG@50: 0.0270
Epoch 11 | NDCG@5: 0.0128 | NDCG@10: 0.0142 | NDCG@20: 0.0215 | NDCG@50: 0.0263
Epoch 12 | NDCG@5: 0.0136 | NDCG@10: 0.0149 | NDCG@20: 0.0212 | NDCG@50: 0.0279
Epoch 13 | NDCG@5: 0.0130 | NDCG@10: 0.0

In [ ]:
history_df = pd.DataFrame(history)
display(history_df)

,epoch,ndcg@5,ndcg@10,ndcg@20,ndcg@50
0,1,0.006796,0.009594,0.011730,0.016566
1,2,0.007862,0.011820,0.014859,0.018938
2,3,0.008879,0.010268,0.014429,0.019469
3,4,0.012211,0.014900,0.015919,0.022331
4,5,0.011631,0.013020,0.015020,0.024138
5,6,0.013823,0.016562,0.019608,0.024466
6,7,0.013302,0.017125,0.017125,0.026038
7,8,0.013897,0.015102,0.018307,0.023213
8,9,0.014551,0.018604,0.020711,0.028978
9,10,0.012574,0.015313,0.019351,0.027043


## Mục 13 - Đánh giá cuối trên validation và test

Báo cáo kết quả trên `VALID`, `TEST`, `TEST_WARM` và `TEST_COLD` để quan sát đồng thời hiệu quả tổng thể và khả năng xử lý cold-start theo item.

In [ ]:
print("=" * 65)
print("FINAL EVALUATION — BASELINE (Tier 1 + Tier 3, NO Tier 2)")
print(f"Config: threshold={POSITIVE_THRESHOLD}, components={NO_COMPONENTS}")
print(f"Item features: {len(all_item_tags)} tags (genre/topic/country/year/duration)")
print(f"User features: {len(all_user_tags)} tags (binary flags + hour/acc_year/weekday)")
print("=" * 65)

for label, inter in [
    ("VALID",     valid_inter),
    ("TEST",      test_inter),
    ("TEST_WARM", test_warm_inter),
    ("TEST_COLD", test_cold_inter),
]:
    metrics = evaluate_metrics(
        model, inter, train_interactions,
        user_feature_matrix, item_feature_matrix,
        k_values=K_VALUES, num_threads=NUM_THREADS,
    )
    print_metrics(metrics, label)

FINAL EVALUATION — BASELINE (Tier 1 + Tier 3, NO Tier 2)
Config: threshold=7, components=64
Item features: 590 tags (genre/topic/country/year/duration)
User features: 19 tags (binary flags + hour/acc_year/weekday)

=== VALID ===
   K |     Prec |   Recall |     NDCG |       HR |      MRR
----------------------------------------------------------
   5 |   0.0042 |   0.0208 |   0.0141 |   0.0208 |   0.0118
  10 |   0.0025 |   0.0250 |   0.0155 |   0.0250 |   0.0124
  20 |   0.0027 |   0.0542 |   0.0226 |   0.0542 |   0.0142
  50 |   0.0017 |   0.0833 |   0.0281 |   0.0833 |   0.0150

=== TEST ===
   K |     Prec |   Recall |     NDCG |       HR |      MRR
----------------------------------------------------------
   5 |   0.0038 |   0.0191 |   0.0153 |   0.0191 |   0.0141
  10 |   0.0027 |   0.0267 |   0.0179 |   0.0267 |   0.0152
  20 |   0.0017 |   0.0344 |   0.0197 |   0.0344 |   0.0157
  50 |   0.0011 |   0.0534 |   0.0233 |   0.0534 |   0.0162

=== TEST_WARM ===
   K |     Prec |   

## Mục 14 - Suy luận: gợi ý cho một người dùng

Minh họa quy trình suy luận top-$N$ item cho một user cụ thể, đồng thời loại bỏ các item đã xuất hiện trong train nhằm phản ánh kịch bản recommendation thực tế.

In [ ]:
def recommend_for_user(user_id, model, dataset, movies_df,
                       user_feature_matrix, item_feature_matrix,
                       train_df, n_recs=10):
    """Gợi ý top-N movies cho 1 user, loại trừ phim đã có trong train."""
    uid_map, _, iid_map, _ = dataset.mapping()
    if user_id not in uid_map:
        print(f"User {user_id} không tồn tại.")
        return pd.DataFrame()

    u_idx   = uid_map[user_id]
    n_items = item_feature_matrix.shape[0]

    scores = model.predict(
        user_ids=u_idx,
        item_ids=np.arange(n_items),
        user_features=user_feature_matrix,
        item_features=item_feature_matrix,
    )

    seen = set(train_df[train_df["id"] == user_id]["movie_id"].values)
    seen_indices = [iid_map[m] for m in seen if m in iid_map]
    scores[seen_indices] = -np.inf

    top_indices   = np.argsort(-scores)[:n_recs]
    idx2movie     = {v: k for k, v in iid_map.items()}
    top_movie_ids = [idx2movie[i] for i in top_indices]

    result = movies_df[movies_df["id"].isin(top_movie_ids)][
        ["id", "original_title", "genres", "year_published", "rate"]
    ].copy()
    result["predicted_score"] = [scores[iid_map[mid]] for mid in result["id"]]
    return result.sort_values("predicted_score", ascending=False)


# Demo
sample_user = train_df["id"].iloc[0]
print(f"[BASELINE] Gợi ý top-10 cho user_id = {sample_user}\n")
recs = recommend_for_user(
    user_id=sample_user, model=model, dataset=dataset,
    movies_df=movies_df,
    user_feature_matrix=user_feature_matrix,
    item_feature_matrix=item_feature_matrix,
    train_df=train_df, n_recs=10,
)
print(recs.to_string(index=False))

[BASELINE] Gợi ý top-10 cho user_id = 103007

    id                                             original_title                                          genres  year_published  rate  predicted_score
491709                                                    Amadeus                                           Drama          1984.0   7.7      -305.394897
272340                                      Bowling for Columbine                                      Documental          2002.0   7.6      -305.965057
800220                                                    Aladdin Animación|Fantástico|Musical|Infantil|Aventuras          1992.0   7.4      -306.021576
565874                                                   Fantasia           Animación|Musical|Fantástico|Infantil          1940.0   7.4      -306.304108
431727 The Assassination of Jesse James By The Coward Robert Ford                                   Western|Drama          2007.0   6.5      -306.308533
640188                              

---
## Tổng kết - So sánh Baseline, Ablation và Proposed

Quy ước:
- **Baseline:** notebook này (`baseline/Lightfm-Baseline.ipynb`)
- **Ablation:** `Lightfm-Proposed-NoUser.ipynb`
- **Proposed:** `Lightfm-Proposed+User.ipynb`

| Thành phần | Baseline | Ablation | Proposed |
|---|---:|---:|---:|
| **Split** | LOO per user | LOO per user | LOO per user |
| **Implicit feedback** | Binary + WARP | Binary + WARP | Binary + WARP |
| **Structured item features** | Có | Có | Có |
| **Dense item features** (Sentence-Transformer, KG, PCA) | Không | Có | Có |
| **Static user-profile features** | Có | Không | Có |
| **User-history features từ train** | Không | Không | Có |
| **Warm/Cold evaluation trên test** | Có | Có | Có |

> Cách diễn giải kết quả: nếu **Ablation** vượt **Baseline**, phần cải thiện có thể quy cho dense item features; nếu **Proposed** tiếp tục vượt **Ablation**, user-history features đã đóng góp thêm ngoài lợi ích của item features.